In [5]:
#INSTALAR LIBRERIAS
!pip install jinja2 weasyprint python-docx -q

In [6]:
#CREAR ESTRUCTURA DE CARPETAS
# Carpeta base del proyecto
from pathlib import Path

BASE = Path('/content/')

# Lista de todas las carpetas que necesitamos
carpetas = [
    'datos/procesados',      # Datasets limpios (Guía 1 y 2)
    'datos/gobernanza',      # Dataset protegido (Guía 4)
    'datos/geo',             # Datasets geográficos (Guía 6)
    'documentos',            # Documentos generales
    'documentos/geo',        # Documentos geográficos
    'documentos/reporte',    # Informes (Guía 7)
    'evidencias',            # Capturas generales
    'evidencias/geo',        # Capturas geográficas
    'evidencias/reporte',    # Gráficos del informe
    'visualizaciones',       # Archivos de Power BI (Guía 5)
    'exportaciones',         # PDFs y DOCXs finales
    'templates',             # Plantillas Jinja2
    'scripts',               # Scripts del pipeline
]

# Crear cada carpeta
for c in carpetas:
    (BASE / c).mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada:")
print(f"Base: {BASE}")
print("\nCarpetas disponibles:")
for c in carpetas:
    print(f"   ├──  {c}/")

Estructura de carpetas creada:
Base: /content

Carpetas disponibles:
   ├──  datos/procesados/
   ├──  datos/gobernanza/
   ├──  datos/geo/
   ├──  documentos/
   ├──  documentos/geo/
   ├──  documentos/reporte/
   ├──  evidencias/
   ├──  evidencias/geo/
   ├──  evidencias/reporte/
   ├──  visualizaciones/
   ├──  exportaciones/
   ├──  templates/
   ├──  scripts/


In [7]:
# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt              # Gráficos estáticos (estable)
from pathlib import Path
from datetime import datetime
from jinja2 import Template
import base64
import json
import warnings
warnings.filterwarnings('ignore')  # Ignorar warnings menores

# CONFIGURAR MATPLOTLIB
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(" Librerías importadas")
print(f"   - pandas: {pd.__version__}")
print(f"   - matplotlib: {plt.matplotlib.__version__}")

# CARGAR DATASETS
df = pd.read_csv(BASE / 'datos' / 'procesados' / 'dataset_limpio_HRTech_People_Analytics.csv')

df_sucursales = pd.read_csv(BASE / 'datos' / 'geo' / 'sucursales_hrtech_guia06.csv')
df_demanda = pd.read_csv(BASE / 'datos' / 'geo' / 'demanda_hrtech_guia06.csv')

print(f"\nDatasets cargados:")
print(f"   - Empleados: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"   - Sucursales: {df_sucursales.shape[0]} filas")
print(f"   - Demanda: {df_demanda.shape[0]} filas")

print(f"\nPrimeras 3 filas del dataset principal:")
print(df.head(3).to_string())

 Librerías importadas
   - pandas: 2.2.3
   - matplotlib: 3.10.0

Datasets cargados:
   - Empleados: 10 filas × 22 columnas
   - Sucursales: 20 filas
   - Demanda: 900 filas

Primeras 3 filas del dataset principal:
   id_empleado  nombre departamento  edad  desempeño  salario  tiempo_empresa_anios estado_empleado  salario_anual  id_capacitacion                  curso  horas  resultado  horas_capacitacion  satisfaccion  horas_extra  salario_por_hora  anios_redondeados  es_antiguo categoria_antiguedad      rango_edad  requiere_capacitacion
0            1     Ana       Ventas    25         85    550.0                     2          Activo         6600.0                1         Excel Avanzado     10   Aprobado                  10             4            5         55.000000                  2       False               Junior     Joven (<30)                  False
1            2    Luis           IT    31         92    900.0                     5          Activo        10800.0             

In [8]:
#CALCULAR KPIs
# NOTA: los KPIs se calculan sobre los 10 colaboradores (activos + inactivos); NO se filtra por estatus.
# El documento (sección 4.2) debe decir lo mismo.
kpis = {
    'total_empleados':     len(df),
    'promedio_desempeno':  df['desempeño'].mean(),
    'promedio_satisfac':   df['satisfaccion'].mean(),
    'total_horas_extra':   df['horas_extra'].sum(),
    'horas_capacitacion':  df['horas'].sum(),

    'empleados_activos':   (df['estado_empleado'] == 'Activo').sum(),
    'empleados_inactivos': (df['estado_empleado'] == 'Inactivo').sum(),
    'masa_salarial_mes':   df['salario'].sum(),
    'masa_salarial_anual': df['salario_anual'].sum()
}

print("KPIs DEL PERIODO")
print("=" * 50)
print(f"1. Total Colaboradores:    {kpis['total_empleados']:>10} empleados")
print(f"2. Promedio de Desempeño:  {kpis['promedio_desempeno']:>10.1f} / 100")
print(f"3. Satisfacción Laboral:   {kpis['promedio_satisfac']:>10.1f} / 5")
print(f"4. Total Horas Extra:      {kpis['total_horas_extra']:>10,} horas")
print(f"5. Horas de Capacitación:  {kpis['horas_capacitacion']:>10,} horas")
print("-" * 50)
print(f"• Datos de Soporte: Nómina Mensual Total de ${kpis['masa_salarial_mes']:,.2f}")
print("=" * 50)

KPIs DEL PERIODO
1. Total Colaboradores:            10 empleados
2. Promedio de Desempeño:        81.9 / 100
3. Satisfacción Laboral:          3.7 / 5
4. Total Horas Extra:              65 horas
5. Horas de Capacitación:          90 horas
--------------------------------------------------
• Datos de Soporte: Nómina Mensual Total de $7,300.00


In [9]:
# ANÁLISIS POR CATEGORÍA (Cursos)
df_categoria = df.groupby('curso').agg(
    salario_total=('salario', 'sum'),
    horas_totales=('horas', 'sum'),
    colaboradores=('id_empleado', 'count')
).reset_index()

# Calculamos el promedio de horas de capacitación por curso
df_categoria['horas_promedio'] = df_categoria['horas_totales'] / df_categoria['colaboradores']
df_categoria = df_categoria.sort_values('colaboradores', ascending=False)

print("ANÁLISIS POR CURSO (CATEGORÍA):")
print(df_categoria.to_string(index=False))


# ANÁLISIS POR CANAL (Departamentos)
df_canal = df.groupby('departamento').agg(
    total_nomina=('salario', 'sum'),
    promedio_desempeño=('desempeño', 'mean'),
    operaciones=('id_empleado', 'count')
).reset_index()

# Calculamos el desempeño relativo por cada dólar invertido en la nómina del departamento
df_canal['desempeño_por_salario'] = (df_canal['promedio_desempeño'] / df_canal['total_nomina']) * 100
df_canal = df_canal.sort_values('total_nomina', ascending=False)

print("\nANÁLISIS POR DEPARTAMENTO (CANAL):")
print(df_canal.to_string(index=False))

# ANÁLISIS POR REGIÓN (Sucursales)
df_region = df_sucursales.groupby('region').agg(
    ventas_totales=('ventas_mes', 'sum'),
    variacion_promedio=('variacion_pct', 'mean'),
    sucursales=('municipio', 'count')
).reset_index()
df_region = df_region.sort_values('ventas_totales', ascending=False)

print("\nANÁLISIS POR REGIÓN: (SUCURSALES)")
print(df_region.to_string(index=False))


ANÁLISIS POR CURSO (CATEGORÍA):
                   curso  salario_total  horas_totales  colaboradores  horas_promedio
       Análisis De Datos          720.0             12              1            12.0
   Comunicación Efectiva          900.0              8              1             8.0
          Excel Avanzado          550.0             10              1            10.0
      Gestión Del Tiempo          950.0              6              1             6.0
  Inteligencia Emocional          800.0              8              1             8.0
               Liderazgo          650.0             12              1            12.0
                Power Bi          700.0             10              1            10.0
Resolución De Conflictos          600.0              8              1             8.0
   Seguridad Informática          680.0             10              1            10.0
       Trabajo En Equipo          750.0              6              1             6.0

ANÁLISIS POR DEPARTAM

In [10]:
# ===================================================================
# GRÁFICO 1: Tendencia laboral (líneas)
# ===================================================================
df_orden = df.sort_values('desempeño')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_orden['desempeño'], df_orden['satisfaccion'],
        marker='o', linewidth=2.5, markersize=10,
        color='#2c7bb6', markerfacecolor='#1a5490')
ax.fill_between(df_orden['desempeño'], df_orden['satisfaccion'], alpha=0.2, color='#2c7bb6')
ax.set_title('Relación de Satisfacción Laboral por Nivel de Desempeño', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Evaluación de Desempeño', fontsize=11)
ax.set_ylabel('Satisfacción Laboral', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=45)

for desempeño, valor in zip(df_orden['desempeño'], df_orden['satisfaccion']):
    ax.annotate(f'{valor:.1f}', (desempeño, valor),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
ruta_tendencia = BASE / 'evidencias' / 'reporte' / 'grafico_tendencia.png'
ruta_tendencia.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(ruta_tendencia, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 1: {ruta_tendencia.name}")

# ===================================================================
# GRÁFICO 2: Colaboradores por curso (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(12, 6))
colores = plt.cm.Blues(np.linspace(0.4, 0.9, len(df_categoria)))
barras = ax.bar(df_categoria['curso'], df_categoria['colaboradores'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)

ax.set_title('Distribución de Colaboradores por Curso de Capacitación', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Cursos Académicos (Categorías)', fontsize=11)
ax.set_ylabel('Cantidad de Empleados', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

ax.set_xticks(range(len(df_categoria)))
ax.set_xticklabels(df_categoria['curso'], rotation=35, ha='right', fontsize=9)

from matplotlib.ticker import MaxNLocator
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

for barra, valor in zip(barras, df_categoria['colaboradores']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{int(valor)}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_categoria = BASE / 'evidencias' / 'reporte' / 'grafico_categorias.png'
plt.savefig(ruta_categoria, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 2: {ruta_categoria.name}")

# ===================================================================
# GRÁFICO 3: Nómina por departamento (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#2D677A', '#2D7A6D', '#324B69']
barras = ax.bar(df_canal['departamento'], df_canal['total_nomina'],
                color=colores[:len(df_canal)], edgecolor='#2c3e50', linewidth=1.5)
ax.set_title('Inversión de Nómina Mensual por Departamento', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Departamento Organizacional (Canal)', fontsize=11)
ax.set_ylabel('Costo Total de Nómina ($)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_canal['total_nomina']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'${valor:,.2f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_canal = BASE / 'evidencias' / 'reporte' / 'grafico_canal.png'
plt.savefig(ruta_canal, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 3: {ruta_canal.name}")

# ===================================================================
# GRÁFICO 4: Presupuesto operativo por región (barras)
# ===================================================================

fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Greens(np.linspace(0.4, 0.9, len(df_region)))
barras = ax.bar(df_region['region'], df_region['ventas_totales'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)
ax.set_title('Ventas Mensuales por Región Geográfica', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Región de El Salvador', fontsize=11)
ax.set_ylabel('Ventas Mensuales ($)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_region['ventas_totales']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'${valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_region = BASE / 'evidencias' / 'reporte' / 'grafico_region.png'
plt.savefig(ruta_region, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 4: {ruta_region.name}")

print("\n¡Los 4 gráficos de Recursos Humanos se generaron correctamente SIN reiniciar Colab!")

------ Gráfico 1: grafico_tendencia.png
------ Gráfico 2: grafico_categorias.png
------ Gráfico 3: grafico_canal.png
------ Gráfico 4: grafico_region.png

¡Los 4 gráficos de Recursos Humanos se generaron correctamente SIN reiniciar Colab!


In [11]:
# VERIFICAR LOS 4 GRÁFICOS
graficos = {
    'Tendencia':   ruta_tendencia,
    'Categorías':  ruta_categoria,
    'Canal':       ruta_canal,
    'Región':      ruta_region,
}

print("VERIFICANDO GRÁFICOS:\n")
for nombre, ruta in graficos.items():
    if ruta.exists():
        tamaño = ruta.stat().st_size / 1024
        print(f"OK {nombre}: {ruta.name} ({tamaño:.1f} KB)")
    else:
        print(f"FAILED {nombre}: NO ENCONTRADO")

# DESCARGAR
from google.colab import files
print("\nDescargando gráficos...")
for nombre, ruta in graficos.items():
    if ruta.exists():
        files.download(str(ruta))

VERIFICANDO GRÁFICOS:

OK Tendencia: grafico_tendencia.png (66.0 KB)
OK Categorías: grafico_categorias.png (96.7 KB)
OK Canal: grafico_canal.png (68.6 KB)
OK Región: grafico_region.png (53.6 KB)

Descargando gráficos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# INSTALAR HARFBUZZ-SUBSET
!apt-get update -qq
!apt-get install -y libharfbuzz-subset0 -qq

# Reinstalar WeasyPrint con soporte para HarfBuzz
!pip install -U weasyprint -q

print("HarfBuzz-Subset instalado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libharfbuzz-subset0:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../libharfbuzz-subset0_8.3.0-2build2_amd64.deb ...
Unpacking libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Setting up libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_

In [13]:
# TEMPLATE HTML CON JINJA2
template_informe = """
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Informe Gerencial - HRTech People Analytics</title>
    <style>
        @page { size: A4; margin: 2cm; }
        :root {
            --primary: #1e3a8a;
            --secondary: #0284c7;
            --accent: #f59e0b;
            --bg-light: #f8fafc;
            --text-main: #1e293b;
            --border-color: #e2e8f0;
        }
        body { font-family: 'Segoe UI', Arial, sans-serif; margin: 0; color: var(--text-main); line-height: 1.5; background-color: #ffffff; font-size: 13px; }

        /* Portada Ejecutiva */
        .portada { text-align: center; margin-bottom: 20px; padding: 20px; background: linear-gradient(135deg, #f0fdf4 0%, #eff6ff 100%); border-radius: 12px; border: 1px solid var(--border-color); }
        .portada h1 { border: none; font-size: 26px; color: var(--primary); margin: 0 0 6px 0; font-weight: 700; }
        .portada h2 { border: none; color: var(--secondary); margin: 0; font-size: 17px; font-weight: 500; }
        .portada-meta { margin-top: 12px; font-size: 12px; color: #64748b; }
        .portada-meta span { margin: 0 8px; }

        /* Encabezados de sección */
        h2 { color: var(--primary); margin-top: 28px; border-bottom: 2px solid var(--primary); padding-bottom: 5px; font-size: 18px; }
        h3 { color: #334155; margin-top: 18px; font-size: 14px; }

        .hallazgo, .kpi-card, table, .bloque-visual { page-break-inside: avoid; }
        .salto { page-break-after: always; }

        /* KPIs Grid */
        .kpi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin: 10px 0; }
        .kpi-card { background: var(--bg-light); padding: 10px 12px; border-left: 4px solid var(--secondary); border-radius: 6px; }
        .kpi-valor { font-size: 20px; font-weight: bold; color: var(--primary); }
        .kpi-label { font-size: 10px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; }

        /* Resumen ejecutivo */
        .resumen-bloque { margin: 8px 0; }
        .resumen-bloque ol, .resumen-bloque ul { margin: 4px 0; padding-left: 22px; }
        .prioridad { font-weight: bold; }

        /* Bloques especiales */
        .hallazgo { background: #fffbeb; padding: 12px 20px; border-left: 5px solid var(--accent); margin: 14px 0; border-radius: 8px; }
        .hallazgo h3 { margin-top: 0; color: #b45309; }
        .hallazgo p { margin: 4px 0; }
        .metodologia { background: #f8fafc; padding: 14px; border-radius: 8px; margin: 14px 0; border: 1px solid var(--border-color); }

        /* Tablas */
        table { border-collapse: collapse; width: 100%; margin: 14px 0; border: 1px solid var(--border-color); }
        th, td { padding: 7px 8px; text-align: left; font-size: 11px; vertical-align: top; }
        th { background: var(--primary); color: white; font-weight: 600; text-transform: uppercase; font-size: 10px; }
        tr:nth-child(even) { background: #f8fafc; }

        /* Imágenes y Pie */
        img { display: block; max-width: 88%; height: auto; margin: 10px auto; border: 1px solid var(--border-color); border-radius: 8px; }
        .pie { font-size: 11px; color: #94a3b8; margin-top: 30px; border-top: 1px solid var(--border-color); padding-top: 10px; text-align: center; }
    </style>
</head>
<body>

<div class="portada">
    <h1>INFORME GERENCIAL DE TALENTO HUMANO</h1>
    <h2>HRTech People Analytics</h2>
    <div class="portada-meta">
        <span><strong>Periodo:</strong> {{ periodo }}</span>
        <span><strong>Fecha:</strong> {{ fecha_generacion }}</span>
        <span><strong>Versión:</strong> {{ version }}</span>
        <span><strong>Equipo:</strong> {{ equipo }}</span>
    </div>
</div>

<!-- ============ 1. RESUMEN EJECUTIVO (una sola página) ============ -->
<div class="salto">
<h2>1. Resumen Ejecutivo</h2>

<div class="resumen-bloque"><strong>Propósito:</strong> Evaluar el desempeño del talento humano, el clima laboral y la distribución territorial de operaciones de HRTech para orientar decisiones de retención, cargas de trabajo y capacitación.</div>
<div class="resumen-bloque"><strong>Periodo analizado:</strong> {{ periodo }}</div>

<div class="resumen-bloque"><strong>KPIs principales:</strong></div>
<div class="kpi-grid">
    <div class="kpi-card"><div class="kpi-label">Total Colaboradores</div><div class="kpi-valor">{{ kpis.total_colaboradores }}</div></div>
    <div class="kpi-card"><div class="kpi-label">Promedio Desempeño</div><div class="kpi-valor">{{ "%.1f"|format(kpis.promedio_desempeno) }}/100</div></div>
    <div class="kpi-card"><div class="kpi-label">Satisfacción Laboral</div><div class="kpi-valor">{{ "%.1f"|format(kpis.promedio_satisfaccion) }}/5.0</div></div>
    <div class="kpi-card"><div class="kpi-label">Total Horas Extra</div><div class="kpi-valor">{{ kpis.total_horas_extra }} hrs</div></div>
    <div class="kpi-card"><div class="kpi-label">Horas de Capacitación</div><div class="kpi-valor">{{ kpis.total_horas_capacitacion }} hrs</div></div>
    <div class="kpi-card"><div class="kpi-label">Índice de Retención</div><div class="kpi-valor">{{ "%.1f"|format(kpis.indice_retencion) }}%</div></div>
</div>

<div class="resumen-bloque"><strong>Hallazgos clave:</strong>
    <ol>
    {% for h in hallazgos %}<li>{{ h.titulo }}.</li>{% endfor %}
    </ol>
</div>

<div class="resumen-bloque"><strong>Recomendaciones priorizadas:</strong>
    <ol>
    {% for r in recomendaciones %}<li><span class="prioridad">[{{ r.prioridad }}]</span> {{ r.accion }}.</li>{% endfor %}
    </ol>
</div>

<div class="resumen-bloque"><strong>Riesgos y limitaciones:</strong>
    <ul>
    {% for l in limitaciones %}<li>{{ l.limitacion }}.</li>{% endfor %}
    </ul>
</div>
</div>

<!-- ============ 2. CONTEXTO Y METODOLOGÍA ============ -->
<h2>2. Contexto y Metodología</h2>
<div class="metodologia">
    <p><strong>Problema empresarial:</strong> La información de personal estaba dispersa en varios archivos y formatos, lo que impedía analizar el desempeño, la inactividad laboral y la efectividad de las capacitaciones.</p>
    <p><strong>Usuarios beneficiados:</strong> Gerencia de Recursos Humanos y analistas de datos.</p>
    <p><strong>Fuentes de datos unificadas:</strong> Registros de empleados (CSV), base transaccional de nómina y desempeño (SQLite) y registros complementarios de formación (JSON).</p>
    <p><strong>Proceso analítico:</strong> Ingesta → Limpieza y homologación (id de empleado, nombres de departamentos, valores negativos) → Análisis exploratorio → Análisis geoespacial (ArcGIS) → Reportería automatizada (Jinja2 + WeasyPrint).</p>
    <p><strong>Alcance y filtros:</strong> Periodo {{ periodo }}, personal activo e inactivo, sucursales agrupadas en tres regiones de El Salvador (Centro, Occidente y Oriente).</p>
</div>

<!-- ============ 3. RESULTADOS GENERALES ============ -->
<h2>3. Resultados Generales</h2>

<h3>3.1 KPIs principales</h3>
<table>
    <thead><tr><th>KPI</th><th>Valor</th><th>Unidad</th><th>Periodo</th></tr></thead>
    <tbody>
        <tr><td>Total colaboradores</td><td>{{ kpis.total_colaboradores }}</td><td>Empleados</td><td>{{ periodo }}</td></tr>
        <tr><td>Promedio de desempeño</td><td>{{ "%.1f"|format(kpis.promedio_desempeno) }}</td><td>Puntos (0-100)</td><td>{{ periodo }}</td></tr>
        <tr><td>Satisfacción laboral</td><td>{{ "%.1f"|format(kpis.promedio_satisfaccion) }}</td><td>Puntos (1-5)</td><td>{{ periodo }}</td></tr>
        <tr><td>Total horas extra</td><td>{{ kpis.total_horas_extra }}</td><td>Horas</td><td>{{ periodo }}</td></tr>
        <tr><td>Horas de capacitación</td><td>{{ kpis.total_horas_capacitacion }}</td><td>Horas</td><td>{{ periodo }}</td></tr>
    </tbody>
</table>

{% for v in visuales %}
<div class="bloque-visual">
<h3>3.{{ loop.index + 1 }} {{ v.titulo }}</h3>
{% if v.imagen %}<img src="{{ v.imagen }}" alt="{{ v.titulo }}">{% endif %}
<p><strong>Lectura:</strong> {{ v.lectura }}</p>
</div>
{% endfor %}

<!-- ============ 4. ANÁLISIS GEOESPACIAL ============ -->
<h2>4. Análisis Geoespacial y Territorial</h2>
{% if ruta_geoespacial %}<img src="{{ ruta_geoespacial }}" alt="Mapa de sucursales - HRTech">{% endif %}
{% if ruta_region %}<img src="{{ ruta_region }}" alt="Ventas por región">{% endif %}
<p><strong>Patrón observado:</strong> {{ lectura_geo }}</p>

<!-- ============ 5. HALLAZGOS ============ -->
<h2>5. Hallazgos Principales</h2>
{% for h in hallazgos %}
<div class="hallazgo">
    <h3>{{ loop.index }}. {{ h.titulo }}</h3>
    <p><strong>Evidencia:</strong> {{ h.evidencia }}</p>
    <p><strong>Interpretación:</strong> {{ h.interpretacion }}</p>
    <p><strong>Impacto organizacional:</strong> {{ h.impacto }}</p>
    <p><strong>Decisión sugerida:</strong> {{ h.decision }}</p>
    <p><em>Limitación asociada: {{ h.limitacion }}</em></p>
</div>
{% endfor %}

<!-- ============ 6. RECOMENDACIONES ============ -->
<h2>6. Matriz de Recomendaciones</h2>
<table>
    <thead>
        <tr><th>#</th><th>Hallazgo</th><th>Acción recomendada</th><th>Prioridad</th><th>Responsable</th><th>Plazo</th><th>KPI de seguimiento</th><th>Riesgo</th></tr>
    </thead>
    <tbody>
        {% for r in recomendaciones %}
        <tr>
            <td>{{ loop.index }}</td>
            <td>{{ r.hallazgo }}</td>
            <td>{{ r.accion }}</td>
            <td><strong>{{ r.prioridad }}</strong></td>
            <td>{{ r.responsable }}</td>
            <td>{{ r.plazo }}</td>
            <td>{{ r.kpi }}</td>
            <td>{{ r.riesgo }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>

<!-- ============ 7. LIMITACIONES ============ -->
<h2>7. Limitaciones</h2>
<table>
    <thead><tr><th>Limitación</th><th>Impacto</th><th>Mitigación</th></tr></thead>
    <tbody>
        {% for l in limitaciones %}
        <tr><td>{{ l.limitacion }}</td><td>{{ l.impacto }}</td><td>{{ l.mitigacion }}</td></tr>
        {% endfor %}
    </tbody>
</table>

<!-- ============ 8. CONCLUSIONES ============ -->
<h2>8. Conclusiones</h2>
<p>El análisis del periodo {{ periodo }} refleja un desempeño general estable en HRTech, con un promedio de {{ "%.1f"|format(kpis.promedio_desempeno) }}/100 y una satisfacción de {{ "%.1f"|format(kpis.promedio_satisfaccion) }}/5.
La prioridad inmediata es rebalancear las cargas de trabajo en {{ depto_alto_he }}, replicar las buenas prácticas de {{ depto_top_desempeno }} en las áreas con menor desempeño y validar el crecimiento observado en la región Oriente antes de ampliar operaciones.</p>

<div class="pie">
    <p>Informe automatizado de Talento Humano - HRTech People Analytics</p>
    <p>Proyecto Integrador - Ingeniería en Desarrollo de Software - Guía Práctica N.º 7</p>
</div>

</body>
</html>
"""

print("Template Jinja2 definido")
print(f"Longitud: {len(template_informe):,} caracteres")


Template Jinja2 definido
Longitud: 10,723 caracteres


In [14]:
# =====================================================================
# CELDA: GENERAR INFORME GERENCIAL (HTML + PDF)
# =====================================================================

# GENERAR INFORME GERENCIAL (HTML + PDF) CON JINJA2 Y WEASYPRINT
import base64
from datetime import datetime
from jinja2 import Template
from google.colab import files
from weasyprint import HTML
from pathlib import Path

BASE = Path('/content/')

# FUNCIÓN PARA CONVERTIR IMAGEN A BASE64 (para incrustarla en el HTML)
def imagen_a_base64(ruta):
    try:
        with open(ruta, 'rb') as f:
            return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"
    except FileNotFoundError:
        print(f"AVISO: no se encontró la imagen {ruta}")
        return ""

# RUTAS DE LOS 4 GRÁFICOS (los mismos de la tabla 4.3.2) Y DEL MAPA
ruta_tendencia = BASE / 'evidencias' / 'reporte' / 'grafico_tendencia.png'
ruta_categoria = BASE / 'evidencias' / 'reporte' / 'grafico_categorias.png'
ruta_canal = BASE / 'evidencias' / 'reporte' / 'grafico_canal.png'
ruta_region = BASE / 'evidencias' / 'reporte' / 'grafico_region.png'
ruta_geoespacial = BASE / 'mapa_geo.png'   # Captura del mapa de ArcGIS (Guía 6), subirla a /content

# ===================================================================
# CALCULAR LOS DATOS DEL INFORME A PARTIR DE LOS DATAFRAMES
# ===================================================================

# 1. Desempeño por departamento
desempeno_depto = df.groupby('departamento')['desempeño'].mean().sort_values(ascending=False)
depto_top_nombre = desempeno_depto.index[0]
depto_top_val = desempeno_depto.iloc[0]
depto_bajo_nombre = desempeno_depto.index[-1]
depto_bajo_val = desempeno_depto.iloc[-1]

# 2. Horas extra por departamento
horas_extra_depto = df.groupby('departamento')['horas_extra'].sum().sort_values(ascending=False)
depto_he_nombre = horas_extra_depto.index[0]
total_he_depto = int(horas_extra_depto.iloc[0])
pct_he_depto = total_he_depto / df['horas_extra'].sum() * 100

# 3. Relación desempeño - satisfacción
correlacion = df['desempeño'].corr(df['satisfaccion'])

# 4. Cursos con más horas
max_horas_curso = df_categoria['horas_totales'].max()
cursos_mas_horas = df_categoria[df_categoria['horas_totales'] == max_horas_curso]['curso'].tolist()

# 5. Nómina por departamento
depto_nomina_nombre = df_canal.iloc[0]['departamento']
depto_nomina_val = df_canal.iloc[0]['total_nomina']

# 6. Sucursales y regiones
ventas_region = df_sucursales.groupby('region')['ventas_mes'].sum()
pct_centro = ventas_region['Centro'] / ventas_region.sum() * 100
suc_crecimiento = df_sucursales.sort_values('variacion_pct', ascending=False).iloc[0]

# KPIs
empleados_activos = (df['estado_empleado'] == 'Activo').sum()
kpis_mapeados = {
    'total_colaboradores': kpis['total_empleados'],
    'promedio_desempeno': kpis['promedio_desempeno'],
    'promedio_satisfaccion': kpis['promedio_satisfac'],
    'total_horas_extra': kpis['total_horas_extra'],
    'total_horas_capacitacion': kpis['horas_capacitacion'],
    'indice_retencion': empleados_activos / kpis['total_empleados'] * 100
}

print("DATOS CALCULADOS PARA EL INFORME")
print("=" * 50)
print(f"Mayor desempeño:     {depto_top_nombre} ({depto_top_val:.1f})")
print(f"Menor desempeño:     {depto_bajo_nombre} ({depto_bajo_val:.1f})")
print(f"Más horas extra:     {depto_he_nombre} ({total_he_depto} h, {pct_he_depto:.0f}% del total)")
print(f"Correlación desempeño-satisfacción: {correlacion:.2f}")
print(f"Cursos con más horas: {', '.join(cursos_mas_horas)} ({max_horas_curso} h)")
print(f"Mayor nómina:        {depto_nomina_nombre} (${depto_nomina_val:,.2f})")
print(f"Ventas región Centro: {pct_centro:.1f}%")
print(f"Mayor crecimiento:   {suc_crecimiento['municipio']} ({suc_crecimiento['variacion_pct']:+.1f}%)")

# ===================================================================
# CONTENIDO DEL INFORME
# ===================================================================
datos_template = {
    'periodo': 'Ciclo Operativo Actual (Octubre 2026)',
    'fecha_generacion': datetime.now().strftime('%d/%m/%Y %H:%M'),
    'version': '1.0',
    'equipo': 'Equipo HRTech People Analytics',
    'kpis': kpis_mapeados,
    'depto_top_desempeno': depto_top_nombre,
    'depto_alto_he': depto_he_nombre,

    # Los 4 visuales de la tabla 4.3.2 (mismo orden)
    'visuales': [
        {
            'titulo': 'Relación entre Desempeño y Satisfacción Laboral',
            'imagen': imagen_a_base64(ruta_tendencia),
            'lectura': (f'A mayor desempeño, mayor satisfacción laboral (correlación de {correlacion:.2f}). Los colaboradores con evaluaciones más bajas son también los menos satisfechos.'
                        if correlacion > 0 else
                        f'No se observa que un mayor desempeño vaya acompañado de mayor satisfacción (correlación de {correlacion:.2f}).')
        },
        {
            'titulo': 'Distribución de Colaboradores por Curso de Capacitación',
            'imagen': imagen_a_base64(ruta_categoria),
            'lectura': f'Los {kpis["total_empleados"]} colaboradores están distribuidos en {len(df_categoria)} cursos distintos (1 por curso). Los cursos con más horas son {", ".join(cursos_mas_horas)} ({max_horas_curso} h cada uno).'
        },
        {
            'titulo': 'Inversión de Nómina Mensual por Departamento',
            'imagen': imagen_a_base64(ruta_canal),
            'lectura': f'{depto_nomina_nombre} es el departamento con mayor costo de nómina mensual (${depto_nomina_val:,.2f}).'
                       + (' También es el de mejor desempeño promedio.' if depto_nomina_nombre == depto_top_nombre else '')
        },
    ],
    'ruta_region': imagen_a_base64(ruta_region),
    'ruta_geoespacial': imagen_a_base64(ruta_geoespacial),
    'lectura_geo': (f'La región Centro concentra el {pct_centro:.1f}% de las ventas mensuales, con San Salvador, Santa Tecla y Soyapango a la cabeza. '
                    f'{suc_crecimiento["municipio"]} ({suc_crecimiento["region"]}) es la sucursal con mayor crecimiento ({suc_crecimiento["variacion_pct"]:+.1f}%), '
                    f'mientras que las sucursales del norte y la zona paracentral muestran menor volumen.'),

    # 3 HALLAZGOS
    'hallazgos': [
        {
            'titulo': f'Alto rendimiento en el departamento de {depto_top_nombre}',
            'evidencia': f'{depto_top_nombre} registra un desempeño promedio de {depto_top_val:.1f} puntos, frente a {kpis["promedio_desempeno"]:.1f} del promedio general y {depto_bajo_val:.1f} de {depto_bajo_nombre}.',
            'interpretacion': 'Las prácticas de trabajo y las competencias técnicas de esta área están bien alineadas con sus metas.',
            'impacto': 'Sirve como modelo de referencia que puede replicarse en las áreas con menor desempeño.',
            'decision': f'Documentar sus buenas prácticas y organizar mentorías con {depto_bajo_nombre}.',
            'limitacion': 'Cada departamento tiene solo 2 colaboradores, por lo que el promedio es sensible a una sola persona.'
        },
        {
            'titulo': f'Concentración de horas extra en el área de {depto_he_nombre}',
            'evidencia': f'{depto_he_nombre} acumula {total_he_depto} de las {kpis["total_horas_extra"]} horas extra del periodo ({pct_he_depto:.0f}% del total).',
            'interpretacion': 'Existe un desbalance en la asignación de tareas y posible saturación operativa.',
            'impacto': 'Eleva el riesgo de fatiga, baja la satisfacción y aumenta la probabilidad de rotación.',
            'decision': 'Redistribuir cargas de trabajo y establecer un tope de horas extra por colaborador.',
            'limitacion': 'El registro no detalla qué proyectos o tareas generaron las horas extra.'
        },
        {
            'titulo': f'Crecimiento territorial en {suc_crecimiento["municipio"]} (región {suc_crecimiento["region"]})',
            'evidencia': f'{suc_crecimiento["municipio"]} presenta ventas de ${suc_crecimiento["ventas_mes"]:,.0f} y una variación de {suc_crecimiento["variacion_pct"]:+.1f}%, la más alta de las 20 sucursales.',
            'interpretacion': 'El crecimiento puede indicar una oportunidad de expansión en la zona oriental.',
            'impacto': 'Podría requerir más personal y capacitación en esa región para sostener el crecimiento.',
            'decision': 'Revisar la evolución de 3 a 6 periodos y la capacidad operativa de la zona antes de invertir.',
            'limitacion': 'Los datos de sucursales son sintéticos y de un solo periodo; requiere validación temporal.'
        }
    ],

    # 3 RECOMENDACIONES (Alta / Media / Baja)
    'recomendaciones': [
        {
            'hallazgo': f'Horas extra en {depto_he_nombre}',
            'accion': f'Rebalancear la carga operativa y establecer topes de horas extra en {depto_he_nombre}',
            'prioridad': 'Alta',
            'responsable': 'Gerencia de Operaciones / RRHH',
            'plazo': '2 semanas',
            'kpi': 'Total de horas extra por departamento',
            'riesgo': 'Afectar metas comerciales en el corto plazo'
        },
        {
            'hallazgo': f'Alto rendimiento en {depto_top_nombre}',
            'accion': f'Replicar las buenas prácticas de {depto_top_nombre} mediante mentorías con {depto_bajo_nombre}',
            'prioridad': 'Media',
            'responsable': 'Talento Humano / Capacitación',
            'plazo': '6 semanas',
            'kpi': 'Desempeño promedio por departamento',
            'riesgo': 'Baja participación de los colaboradores'
        },
        {
            'hallazgo': f'Crecimiento en {suc_crecimiento["municipio"]}',
            'accion': f'Evaluar la capacidad operativa y de personal en la región {suc_crecimiento["region"]} antes de ampliar',
            'prioridad': 'Baja',
            'responsable': 'Gerencia de Operaciones',
            'plazo': '3 meses',
            'kpi': 'Variación % de ventas por sucursal',
            'riesgo': 'Que el crecimiento no se sostenga en el tiempo'
        }
    ],

    # 3 LIMITACIONES (Limitación / Impacto / Mitigación)
    'limitaciones': [
        {
            'limitacion': f'Muestra pequeña ({kpis["total_empleados"]} colaboradores)',
            'impacto': 'Los promedios por departamento cambian mucho con una sola persona',
            'mitigacion': 'Ampliar la base con más colaboradores y periodos'
        },
        {
            'limitacion': 'Un solo periodo analizado (octubre 2026)',
            'impacto': 'No se pueden observar tendencias ni estacionalidad',
            'mitigacion': 'Acumular un histórico mensual de indicadores'
        },
        {
            'limitacion': 'Datos de sucursales sintéticos',
            'impacto': 'Los hallazgos territoriales son ilustrativos',
            'mitigacion': 'Validar con datos reales de operación de cada sucursal'
        }
    ]
}

# ===================================================================
# GENERAR HTML Y PDF
# ===================================================================
ruta_plantilla = BASE / 'templates' / 'plantilla_hrtech.html'
ruta_html = BASE / 'documentos' / 'reporte' / 'informe_hrtech.html'
ruta_pdf = BASE / 'documentos' / 'reporte' / 'informe_hrtech.pdf'
ruta_plantilla.parent.mkdir(parents=True, exist_ok=True)
ruta_html.parent.mkdir(parents=True, exist_ok=True)

# Guardar la plantilla HTML base
with open(ruta_plantilla, 'w', encoding='utf-8') as f:
    f.write(template_informe)
print(f"\nPlantilla guardada en: {ruta_plantilla}")

# Renderizar template con los datos
template = Template(template_informe)
html_renderizado = template.render(**datos_template)

with open(ruta_html, 'w', encoding='utf-8') as f:
    f.write(html_renderizado)
print(f"HTML generado en: {ruta_html}")

# Convertir a PDF con WeasyPrint
HTML(string=html_renderizado).write_pdf(str(ruta_pdf))
print(f"PDF generado en: {ruta_pdf}")
print(f"Tamaño del archivo: {ruta_pdf.stat().st_size / 1024:.1f} KB")

# Descargar el PDF
files.download(str(ruta_pdf))
print("Descarga iniciada.")


DATOS CALCULADOS PARA EL INFORME
Mayor desempeño:     IT (93.5)
Menor desempeño:     Ventas (75.0)
Más horas extra:     Ventas (19 h, 29% del total)
Correlación desempeño-satisfacción: 0.97
Cursos con más horas: Análisis De Datos, Liderazgo (12 h)
Mayor nómina:        IT ($1,850.00)
Ventas región Centro: 61.4%
Mayor crecimiento:   San Miguel (+9.4%)
AVISO: no se encontró la imagen /content/mapa_geo.png

Plantilla guardada en: /content/templates/plantilla_hrtech.html
HTML generado en: /content/documentos/reporte/informe_hrtech.html
PDF generado en: /content/documentos/reporte/informe_hrtech.pdf
Tamaño del archivo: 294.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada.


In [15]:
# GENERAR INFORME GERENCIAL EN DOCX (python-docx) - mismos datos que el HTML/PDF
from docx import Document
from docx.shared import Pt, Cm, Inches

def docx_tabla(doc, encabezados, filas):
    t = doc.add_table(rows=1, cols=len(encabezados))
    t.style = 'Light Grid Accent 1'
    for j, h in enumerate(encabezados):
        celda = t.rows[0].cells[j]
        celda.text = ''
        celda.paragraphs[0].add_run(str(h)).bold = True
    for fila in filas:
        cs = t.add_row().cells
        for j, v in enumerate(fila):
            cs[j].text = str(v)
    return t

def docx_imagen(doc, ruta, ancho=Inches(6)):
    if Path(ruta).exists():
        doc.add_picture(str(ruta), width=ancho)
    else:
        doc.add_paragraph(f'[Imagen no disponible: {Path(ruta).name}]')

docx = Document()
for sec in docx.sections:
    sec.left_margin = sec.right_margin = sec.top_margin = sec.bottom_margin = Cm(2.5)
docx.styles['Normal'].font.name = 'Arial'
docx.styles['Normal'].font.size = Pt(11)

k = datos_template['kpis']

docx.add_heading('INFORME GERENCIAL DE TALENTO HUMANO', 0)
docx.add_paragraph('HRTech People Analytics')
docx.add_paragraph(f"Periodo: {datos_template['periodo']}  |  Versión: {datos_template['version']}  |  {datos_template['equipo']}")

# 1. Resumen ejecutivo
docx.add_heading('1. Resumen Ejecutivo', 1)
p = docx.add_paragraph(); p.add_run('Propósito: ').bold = True
p.add_run('Evaluar el desempeño del talento humano, el clima laboral y la distribución territorial de operaciones de HRTech para orientar decisiones de retención, cargas de trabajo y capacitación.')
p = docx.add_paragraph(); p.add_run('Periodo analizado: ').bold = True; p.add_run(datos_template['periodo'])
docx.add_paragraph().add_run('KPIs principales:').bold = True
for etiqueta, valor in [
    ('Total colaboradores', f"{k['total_colaboradores']}"),
    ('Promedio de desempeño', f"{k['promedio_desempeno']:.1f}/100"),
    ('Satisfacción laboral', f"{k['promedio_satisfaccion']:.1f}/5"),
    ('Total horas extra', f"{k['total_horas_extra']} h"),
    ('Horas de capacitación', f"{k['total_horas_capacitacion']} h"),
    ('Índice de retención', f"{k['indice_retencion']:.1f}%"),
]:
    docx.add_paragraph(f'{etiqueta}: {valor}', style='List Bullet')
docx.add_paragraph().add_run('Hallazgos clave:').bold = True
for h in datos_template['hallazgos']:
    docx.add_paragraph(h['titulo'], style='List Number')
docx.add_paragraph().add_run('Recomendaciones priorizadas:').bold = True
for r in datos_template['recomendaciones']:
    docx.add_paragraph(f"[{r['prioridad']}] {r['accion']}", style='List Number')
docx.add_paragraph().add_run('Riesgos y limitaciones:').bold = True
for l in datos_template['limitaciones']:
    docx.add_paragraph(l['limitacion'], style='List Bullet')

# 2. Contexto y metodología
docx.add_heading('2. Contexto y Metodología', 1)
for etq, txt in [
    ('Problema empresarial', 'La información de personal estaba dispersa en varios archivos y formatos, lo que impedía analizar el desempeño, la inactividad laboral y la efectividad de las capacitaciones.'),
    ('Usuarios beneficiados', 'Gerencia de Recursos Humanos y analistas de datos.'),
    ('Fuentes de datos', 'empleados.csv, empresa_hr_analytics.db y datos_adicionales.json.'),
    ('Alcance y filtros', f"Periodo {datos_template['periodo']}; personal activo e inactivo ({k['total_colaboradores']} colaboradores); sucursales agrupadas en tres regiones de El Salvador (Centro, Occidente y Oriente)."),
]:
    p = docx.add_paragraph(); p.add_run(f'{etq}: ').bold = True; p.add_run(txt)

# 3. Resultados generales
docx.add_heading('3. Resultados Generales', 1)
docx_tabla(docx, ['KPI', 'Valor', 'Unidad', 'Periodo'], [
    ['Total colaboradores', k['total_colaboradores'], 'Empleados', datos_template['periodo']],
    ['Promedio de desempeño', f"{k['promedio_desempeno']:.1f}", 'Puntos (0-100)', datos_template['periodo']],
    ['Satisfacción laboral', f"{k['promedio_satisfaccion']:.1f}", 'Puntos (1-5)', datos_template['periodo']],
    ['Total horas extra', k['total_horas_extra'], 'Horas', datos_template['periodo']],
    ['Horas de capacitación', k['total_horas_capacitacion'], 'Horas', datos_template['periodo']],
])
rutas_visuales = [ruta_tendencia, ruta_categoria, ruta_canal]
for n, (v, ruta) in enumerate(zip(datos_template['visuales'], rutas_visuales), start=1):
    docx.add_heading(f"3.{n} {v['titulo']}", 2)
    docx_imagen(docx, ruta)
    p = docx.add_paragraph(); p.add_run('Lectura: ').bold = True; p.add_run(v['lectura'])

# 4. Análisis geoespacial
docx.add_heading('4. Análisis Geoespacial y Territorial', 1)
docx_imagen(docx, ruta_geoespacial)
docx_imagen(docx, ruta_region)
p = docx.add_paragraph(); p.add_run('Patrón observado: ').bold = True; p.add_run(datos_template['lectura_geo'])

# 5. Hallazgos
docx.add_heading('5. Hallazgos Principales', 1)
for n, h in enumerate(datos_template['hallazgos'], start=1):
    docx.add_heading(f"{n}. {h['titulo']}", 2)
    for etq, clave in [('Evidencia', 'evidencia'), ('Interpretación', 'interpretacion'),
                       ('Impacto organizacional', 'impacto'), ('Decisión sugerida', 'decision'),
                       ('Limitación asociada', 'limitacion')]:
        p = docx.add_paragraph(); p.add_run(f'{etq}: ').bold = True; p.add_run(h[clave])

# 6. Recomendaciones
docx.add_heading('6. Matriz de Recomendaciones', 1)
docx_tabla(docx, ['#', 'Hallazgo', 'Acción', 'Prioridad', 'Responsable', 'Plazo', 'KPI', 'Riesgo'],
           [[n, r['hallazgo'], r['accion'], r['prioridad'], r['responsable'], r['plazo'], r['kpi'], r['riesgo']]
            for n, r in enumerate(datos_template['recomendaciones'], start=1)])

# 7. Limitaciones
docx.add_heading('7. Limitaciones', 1)
docx_tabla(docx, ['Limitación', 'Impacto', 'Mitigación'],
           [[l['limitacion'], l['impacto'], l['mitigacion']] for l in datos_template['limitaciones']])

# 8. Conclusiones
docx.add_heading('8. Conclusiones', 1)
docx.add_paragraph(
    f"El análisis del periodo {datos_template['periodo']} refleja un desempeño general estable en HRTech, "
    f"con un promedio de {k['promedio_desempeno']:.1f}/100 y una satisfacción de {k['promedio_satisfaccion']:.1f}/5. "
    f"La prioridad inmediata es rebalancear las cargas de trabajo en {datos_template['depto_alto_he']}, "
    f"replicar las buenas prácticas de {datos_template['depto_top_desempeno']} en las áreas con menor desempeño "
    "y validar el crecimiento observado en la región Oriente antes de ampliar operaciones.")

ruta_docx = BASE / 'documentos' / 'reporte' / 'informe_hrtech.docx'
docx.save(str(ruta_docx))
print(f'DOCX generado en: {ruta_docx}')
try:
    files.download(str(ruta_docx))
except Exception as e:
    print('Descarga omitida:', e)

DOCX generado en: /content/documentos/reporte/informe_hrtech.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# METADATA DE TRAZABILIDAD
import json
from datetime import datetime

metadata = {
    'version': '1.0',
    'fecha_generacion': datetime.now().isoformat(),
    'periodo_analizado': datos_template['periodo'],
    'fuentes_datos': [
        'empleados.csv',
        'empresa_hr_analytics.db',
        'datos_adicionales.json'
    ],
    'filtros_aplicados': {
        'periodo': datos_template['periodo'],
        'plausibilidad_territorial': {
            'latitud_rango': [13.0, 15.0],
            'longitud_rango': [-91.0, -87.0],
            'pais': 'El Salvador'
        },
        'estatus_empleado': 'Activos e Inactivos',
        'excluir_registros_invalidos': True
    },
    'indicadores_clave': {
        'total_colaboradores': int(kpis_mapeados['total_colaboradores']),
        'promedio_desempeno': float(kpis_mapeados['promedio_desempeno']),
        'promedio_satisfaccion': float(kpis_mapeados['promedio_satisfaccion']),
        'total_horas_extra': int(kpis_mapeados['total_horas_extra']),
        'total_horas_capacitacion': int(kpis_mapeados['total_horas_capacitacion']),
        'indice_retencion_pct': float(kpis_mapeados['indice_retencion'])
    },
        'archivos_generados': [
        'informe_hrtech.pdf',
        'informe_hrtech.html',
        'grafico_tendencia.png',
        'grafico_categorias.png',
        'grafico_canal.png',
        'grafico_region.png',
        'metadatos_hrtech.json'
    ],
    'generado_por': 'generar_reporte_hrtech.py'
}

# Configurar ruta de guardado en la estructura del proyecto
ruta_metadata = BASE / 'documentos' / 'hrtech' / 'metadatos_hrtech.json'
ruta_metadata.parent.mkdir(parents=True, exist_ok=True)

ruta_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print("Metadata de trazabilidad guardada con éxito:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# Descargar archivo JSON automáticamente en Google Colab
from google.colab import files
files.download(str(ruta_metadata))
print("Metadatos descargados correctamente.")

Metadata de trazabilidad guardada con éxito:
{
  "version": "1.0",
  "fecha_generacion": "2026-11-18T02:05:34.849251",
  "periodo_analizado": "Ciclo Operativo Actual (Octubre 2026)",
  "fuentes_datos": [
    "empleados.csv",
    "empresa_hr_analytics.db",
    "datos_adicionales.json"
  ],
  "filtros_aplicados": {
    "periodo": "Ciclo Operativo Actual (Octubre 2026)",
    "plausibilidad_territorial": {
      "latitud_rango": [
        13.0,
        15.0
      ],
      "longitud_rango": [
        -91.0,
        -87.0
      ],
      "pais": "El Salvador"
    },
    "estatus_empleado": "Activos e Inactivos",
    "excluir_registros_invalidos": true
  },
  "indicadores_clave": {
    "total_colaboradores": 10,
    "promedio_desempeno": 81.9,
    "promedio_satisfaccion": 3.7,
    "total_horas_extra": 65,
    "total_horas_capacitacion": 90,
    "indice_retencion_pct": 70.0
  },
  "archivos_generados": [
    "informe_hrtech.pdf",
    "informe_hrtech.html",
    "grafico_tendencia.png",
    "graf

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Metadatos descargados correctamente.


In [17]:
# MATRIZ DE RECOMENDACIONES Y MATRIZ DE TRAZABILIDAD EN EXCEL
import pandas as pd

carpeta_rep = BASE / 'documentos' / 'reporte'
carpeta_rep.mkdir(parents=True, exist_ok=True)

# ---- Matriz de recomendaciones (Anexo B) ----
df_reco = pd.DataFrame(datos_template['recomendaciones'])
df_reco.insert(0, 'n', range(1, len(df_reco) + 1))
df_reco.columns = ['N.º', 'Hallazgo', 'Acción recomendada', 'Prioridad', 'Responsable',
                   'Plazo', 'KPI de seguimiento', 'Riesgo']
ruta_reco = carpeta_rep / 'matriz_recomendaciones_HRTech.xlsx'
df_reco.to_excel(ruta_reco, index=False)

# hallazgo 0 (alto rendimiento) -> recomendación 1; hallazgo 1 (horas extra) -> recomendación 0; hallazgo 2 -> 2
reco_de_hallazgo = {0: 1, 1: 0, 2: 2}
fuentes_hallazgo = ['dataset_limpio_HRTech_People_Analytics.csv',
                    'dataset_limpio_HRTech_People_Analytics.csv',
                    'sucursales_hrtech_guia06.csv']

filas_traza = []
for i, h in enumerate(datos_template['hallazgos']):
    r = datos_template['recomendaciones'][reco_de_hallazgo[i]]
    filas_traza.append([f'H{i+1}', h['titulo'], h['evidencia'], fuentes_hallazgo[i],
                        f"R{reco_de_hallazgo[i]+1}", r['accion'], r['prioridad'], r['kpi']])
df_traza_h = pd.DataFrame(filas_traza, columns=[
    'ID hallazgo', 'Hallazgo', 'Evidencia', 'Fuente de datos',
    'ID recomendación', 'Acción recomendada', 'Prioridad', 'KPI de seguimiento'])

df_traza_k = pd.DataFrame([
    ['Total colaboradores', kpis_mapeados['total_colaboradores'], 'dataset_limpio_HRTech_People_Analytics.csv', "len(df)"],
    ['Promedio de desempeño', round(kpis_mapeados['promedio_desempeno'], 2), 'dataset_limpio_HRTech_People_Analytics.csv', "df['desempeño'].mean()"],
    ['Satisfacción laboral', round(kpis_mapeados['promedio_satisfaccion'], 2), 'dataset_limpio_HRTech_People_Analytics.csv', "df['satisfaccion'].mean()"],
    ['Total horas extra', int(kpis_mapeados['total_horas_extra']), 'dataset_limpio_HRTech_People_Analytics.csv', "df['horas_extra'].sum()"],
    ['Horas de capacitación', int(kpis_mapeados['total_horas_capacitacion']), 'dataset_limpio_HRTech_People_Analytics.csv', "df['horas'].sum()"],
    ['Índice de retención (%)', round(kpis_mapeados['indice_retencion'], 1), 'dataset_limpio_HRTech_People_Analytics.csv', "activos / total * 100"],
], columns=['KPI', 'Valor', 'Fuente', 'Cálculo'])

df_traza_a = pd.DataFrame({
    'Archivo generado': metadata['archivos_generados'],
    'Versión': metadata['version'],
    'Fecha de generación': metadata['fecha_generacion'],
})

ruta_traza = carpeta_rep / 'matriz_trazabilidad_HRTech.xlsx'
with pd.ExcelWriter(ruta_traza) as xl:
    df_traza_h.to_excel(xl, sheet_name='hallazgo_a_recomendacion', index=False)
    df_traza_k.to_excel(xl, sheet_name='kpi_a_fuente', index=False)
    df_traza_a.to_excel(xl, sheet_name='archivos_generados', index=False)

print('Matriz de recomendaciones:', ruta_reco)
print('Matriz de trazabilidad:   ', ruta_traza)
try:
    files.download(str(ruta_reco)); files.download(str(ruta_traza))
except Exception as e:
    print('Descarga omitida:', e)

Matriz de recomendaciones: /content/documentos/reporte/matriz_recomendaciones_HRTech.xlsx
Matriz de trazabilidad:    /content/documentos/reporte/matriz_trazabilidad_HRTech.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# CREAR GITHUB ACTIONS WORKFLOW PARA HRTECH
workflow = """
name: Generar Informe Automático HRTech

on:
  schedule:
    - cron: '0 8 * * 1'  # Todos los lunes a las 8 AM
  workflow_dispatch:

jobs:
  generar-informe:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Instalar dependencias del sistema para WeasyPrint
        run: |
          sudo apt-get update
          sudo apt-get install -y libpango-1.0-0 libpangoft2-1.0-0 libcairo2
      - name: Instalar dependencias de Python
        run: pip install pandas matplotlib jinja2 weasyprint pyspark
      - name: Generar informe HRTech
        run: python scripts/generar_reporte_hrtech.py
      - name: Subir artefactos
        uses: actions/upload-artifact@v3
        with:
          name: informe-hrtech-generado
          path: documentos/hrtech/
"""

ruta_workflow = BASE / '.github' / 'workflows'
ruta_workflow.mkdir(parents=True, exist_ok=True)
(ruta_workflow / 'generar_informe.yml').write_text(workflow.strip(), encoding='utf-8')

print("Workflow creado")
print(f"Ruta: {ruta_workflow / 'generar_informe.yml'}")

Workflow creado
Ruta: /content/.github/workflows/generar_informe.yml


In [19]:
# VER TODOS LOS ARCHIVOS (INCLUYENDO OCULTOS)
print("TODOS LOS ARCHIVOS DE evaluacion_practica2/:\n")
!ls -la /content/

print("\nCONTENIDO DE LA CARPETA .github/:\n")
!ls -la /content/.github/

print("\nCONTENIDO DE .github/workflows/:\n")
!ls -la /content/.github/workflows/

TODOS LOS ARCHIVOS DE evaluacion_practica2/:

total 48
drwxr-xr-x 1 root root 4096 Nov 18 02:05 .
drwxr-xr-x 1 root root 4096 Nov 18 00:32 ..
drwxr-xr-x 4 root root 4096 Oct 26 22:18 .config
drwxr-xr-x 5 root root 4096 Nov 18 02:03 datos
drwxr-xr-x 5 root root 4096 Nov 18 02:05 documentos
drwxr-xr-x 4 root root 4096 Nov 18 02:03 evidencias
drwxr-xr-x 2 root root 4096 Nov 18 02:03 exportaciones
drwxr-xr-x 3 root root 4096 Nov 18 02:05 .github
drwxr-xr-x 1 root root 4096 Oct 26 22:18 sample_data
drwxr-xr-x 2 root root 4096 Nov 18 02:03 scripts
drwxr-xr-x 2 root root 4096 Nov 18 02:05 templates
drwxr-xr-x 2 root root 4096 Nov 18 02:03 visualizaciones

CONTENIDO DE LA CARPETA .github/:

total 12
drwxr-xr-x 3 root root 4096 Nov 18 02:05 .
drwxr-xr-x 1 root root 4096 Nov 18 02:05 ..
drwxr-xr-x 2 root root 4096 Nov 18 02:05 workflows

CONTENIDO DE .github/workflows/:

total 12
drwxr-xr-x 2 root root 4096 Nov 18 02:05 .
drwxr-xr-x 3 root root 4096 Nov 18 02:05 ..
-rw-r--r-- 1 root root  868 No

In [20]:
# DESCARGAR EL WORKFLOW PARA VERIFICARLO EN TU PC
from google.colab import files

ruta_workflow = BASE / '.github' / 'workflows' / 'generar_informe.yml'
files.download(str(ruta_workflow))

print("Workflow descargado")
print("   Ábrelo con el Bloc de notas o VS Code")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Workflow descargado
   Ábrelo con el Bloc de notas o VS Code
